# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
meta = dataset.metadata

print(f"Dataset Title: {meta.name}")
print(f"Description: {meta.description}")
print(f"Version: {getattr(meta, 'version', 'N/A')}")
print(f"Authors: {getattr(meta, 'author', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs. The Croissant `@id` uniquely identifies each `recordSet` and each field.

Let's examine the available record sets and print their `@id`s and fields.

In [ ]:
# Explore record sets and their fields
record_sets = getattr(meta, 'recordSet', [])
if not record_sets:
    print('No record sets found. The metadata may define contents differently. Trying .records() for available record sets...')

# mlcroissant lets you enumerate available record_set ids
record_set_ids = dataset.record_set_ids()
print(f'Found {len(record_set_ids)} record sets:')
for rset_id in record_set_ids:
    print(f'  RecordSet @id: {rset_id}')
    record_set = dataset.get_record_set(rset_id)
    # Fields
    if hasattr(record_set, 'fields'):
        field_ids = [f'@id: {f["@id"]}, name: {f["name"]}' for f in record_set.fields.values()]
        print('    Fields:')
        for fid in field_ids:
            print('      ', fid)
    else:
        print('    No fields listed.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.


In [ ]:
# List record sets
record_set_ids = dataset.record_set_ids()
dataframes = {}

# We choose the first record set for demonstration; you can select by @id
selected_record_set_id = record_set_ids[0]
print(f'Loading records from record set: {selected_record_set_id}')

records = list(dataset.records(record_set=selected_record_set_id))
df = pd.DataFrame(records)
dataframes[selected_record_set_id] = df

print(f'Columns in {selected_record_set_id}:')
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For demonstration, select numeric and categorical fields by their `@id`.

In [ ]:
# Identify numeric fields in the DataFrame
df = dataframes[selected_record_set_id]
numeric_candidates = df.select_dtypes(include='number').columns.tolist()
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]  # For demonstration
    print(f'Numeric field selected: {numeric_field_id}')
else:
    print('No numeric fields identified, please check data.')
    numeric_field_id = None

# Filtering and normalization if available
if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f'Filtered records with {numeric_field_id} > {threshold:.2f}:')
    print(filtered_df.head())
    normalized_col = f'{numeric_field_id}_normalized'
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f'Normalized {numeric_field_id} for filtered records:')
    print(filtered_df[[numeric_field_id, normalized_col]].head())
else:
    print('Skipping filtering; no numeric field found.')

# Try grouping by first non-numeric categorical field
group_candidates = df.select_dtypes(include='object').columns.tolist()
group_field_id = None
for col in group_candidates:
    if col != numeric_field_id and df[col].nunique() < len(df) // 2:
        group_field_id = col
        break

if group_field_id is not None and numeric_field_id is not None:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f'Grouped data by {group_field_id}:')
    print(grouped_df.head())
else:
    print('No suitable field found for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and not df.empty:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, color='slateblue')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If group_field_id exists, boxplot
    if group_field_id is not None:
        plt.figure(figsize=(12, 6))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.show()
else:
    print('Not enough numeric data to visualize.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


In this notebook, we loaded and explored the FAIR² dataset _Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells_ using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

- We listed the available record sets and fields by their `@id`.
- We extracted tabular data into a dataframe, performed simple filtering and normalization for a numeric field, and demonstrated basic grouping by a categorical field.
- Visualizations provided an initial sense of data distribution and potential grouping effects.

Further analysis may involve more detailed examination of the mass spectrometry protein data using protein accession numbers (`@id`), molecular weights, peptides, or experiment conditions as defined by the Croissant schema.
